In [1]:
# Install required packages
!pip install streamlit PyMuPDF google-genai python-dotenv

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 24.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.1/24.1 MB 18.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 44.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.1/79.1 kB 3.4 MB/s eta 0:00:00


In [3]:
# Set your API key (you can also use Colab secrets)
from google.colab import userdata
import os
os.environ['GEMINI_API_KEY'] = userdata.get('GEMINI_API_KEY')

In [5]:
# Create the app.py file
%%writefile app.py
# [Paste the complete code here]
import re
import fitz
import json
import streamlit as st
from google import genai
from google.genai import types
from io import BytesIO
import traceback
from datetime import datetime
import os
import sys

# CONFIGURATION
# Load API key - for Colab, you can set this as a secret or environment variable
API_KEY = os.getenv('GEMINI_API_KEY')  # Replace with your actual API key
MODEL_CTX = "gemini-2.0-flash"
MODEL_MAP = "gemini-2.0-flash"

REFERRAL_PROMPT = """
pdf1 is a referral package for a patient and pdf2 is a Prior Authorization (PA) form. pdf1 contains all the details of the patient... Some pages of pdf2 consists of all the questions that needs to be answered inferring from the details in pdf1...
Go through entire referral package and extract answers to as many questions in PA form as possible.
Go through every page in the referral package carefully and extract all the information by visually examining.
Wrap all the patient field in a single top-level object. For example:
{
"patient_info": {
  "First_Name": "...",
  "Last_Name": "...",
  "Date_of_Birth": "...",
}
}
"""

# Initialize Gemini client
try:
    client = genai.Client(api_key=API_KEY)
    st.sidebar.success("Gemini API configured successfully")
except Exception as e:
    client = None
    st.sidebar.error(f"Gemini API configuration failed: {e}")

# UTILITIES
def extract_json(text):
    start = text.find("{")
    end = text.rfind("}")
    if start == -1 or end == -1:
        raise ValueError("No JSON Object found")
    raw = text[start:end+1]
    raw = re.sub(r'(?m)^\s*([A-Za-z0-9_]+)\s*:', r'"\1":', raw)
    raw = re.sub(r',\s*([}\]])', r'\1', raw)
    return raw

def extract_patient_info(referral_bytes, pa_bytes):
    """Runs the user provided Gemini prompt to extract patient info"""
    if not client:
        raise Exception("Gemini client not initialized. Please check your API key.")

    pdf1 = types.Part.from_bytes(data=referral_bytes, mime_type='application/pdf')
    pdf2 = types.Part.from_bytes(data=pa_bytes, mime_type='application/pdf')
    response = client.models.generate_content(
        model=MODEL_CTX,
        contents=[pdf1, pdf2, REFERRAL_PROMPT]
    ).text
    print(response)
    raw = extract_json(response)
    return json.loads(raw)

def make_page_part(pdf_bytes, page_no):
    """
    Create a one-page PDF part from the given PDF bytes.
    Tries to copy the form page; on XRef errors, falls back to image-based PDF.
    """
    src = fitz.open(stream=pdf_bytes, filetype='pdf')
    try:
        dst = fitz.open()
        dst.insert_pdf(src, from_page=page_no-1, to_page=page_no-1)
        for w in dst[0].widgets() or []:
            dst[0].delete_widget(w)
        buf = BytesIO()
        dst.save(buf)
        src.close()
        dst.close()
        return types.Part.from_bytes(data=buf.getvalue(), mime_type='application/pdf')
    except Exception:
        page = src[page_no-1]
        pix = page.get_pixmap()
        new_pdf = fitz.open()
        rect = page.rect
        new_page = new_pdf.new_page(width=rect.width, height=rect.height)
        new_page.insert_image(rect, pixmap=pix)
        buf = BytesIO()
        new_pdf.save(buf)
        src.close()
        new_pdf.close()
        return types.Part.from_bytes(data=buf.getvalue(), mime_type='application/pdf')

def extract_fields_with_positions(pdf_bytes):
    doc = fitz.open(stream=pdf_bytes, filetype='pdf')
    fields = []
    for page_num, page in enumerate(doc, start=1):
        for w in page.widgets() or []:
            fields.append({
                "name": w.field_name,
                "type": "checkbox" if w.field_type == fitz.PDF_WIDGET_TYPE_CHECKBOX else "text",
                "value": w.field_value,
                "page": page_num,
                "rect": list(map(float, w.rect)),
            })
    doc.close()
    return fields

def get_field_context_for_page(pa_bytes, page_no, page_fields):
    """Extract context for fields on a specific page"""
    if not client:
        return []

    page_part = make_page_part(pa_bytes, page_no)

    prompt_ctx = f"""
    You're annotating page {page_no} of a medical Prior Authorization form.
    Given:
        - This page's form fields (id, type, rect).
        - Actual Prior Authorization (PA) form page.

    For each field:
    Add question and context fields to already existing field objects.
    Move sequentially through the page for each field along with fields info attached.

    Generate the best possible context around that field object in 25 words.
    Getting the correct context for the correct field is extremely important for correct mapping.

    Return a JSON array for each field with:
    - For each form field object add its question corresponding to it
    - Also indicate the context in which the question is asked.

    Each output JSON object should only contain the fields - name, page, question, context in the following format:
    {{"name": "T67", "page": 2, "question": "Patient's primary diagnosis", "context": "Medical condition requiring treatment authorization"}}

    Return all the objects as a JSON array inside [].
    Only return valid JSON.

    Here are the fields:
    {json.dumps(page_fields, indent=2)}
    """

    try:
        response = client.models.generate_content(
            model=MODEL_CTX,
            contents=[page_part, prompt_ctx]
        ).text
        print(f"Context response for page {page_no}: {response}")

        # Try to extract JSON array
        start = response.find("[")
        end = response.rfind("]")
        if start != -1 and end != -1:
            raw = response[start:end+1]
            return json.loads(raw)
        else:
            # Fallback to object extraction
            raw = extract_json(response)
            data = json.loads(raw)
            return [data] if isinstance(data, dict) else data

    except Exception as e:
        print(f"Error getting context for page {page_no}: {e}")
        return []

def map_patient_info_to_fields(patient_info, field_contexts):
    """Map extracted patient information to form fields using AI"""
    if not client:
        return {}

    mapping_prompt = f"""
    You are mapping extracted patient information to Prior Authorization form fields.

    Patient Information:
    {json.dumps(patient_info, indent=2)}

    Form Fields with Context:
    {json.dumps(field_contexts, indent=2)}

    Instructions:
    1. For each form field, determine if any patient information matches or answers the field's question
    2. Consider field context to understand what type of information is needed
    3. Handle conditional logic - some fields may be mutually exclusive
    4. For checkbox fields, return "Yes" or "No" or leave empty if uncertain
    5. For text fields, return the appropriate value from patient info or leave empty if no match
    6. Be conservative - only fill fields you're confident about

    Return a JSON object mapping field names to their values:
    {{
        "field_name": "field_value",
        "T1": "John",
        "T2": "Doe",
        "CB1": "Yes",
        "CB2": ""
    }}

    Only return valid JSON with field mappings.
    """

    try:
        response = client.models.generate_content(
            model=MODEL_MAP,
            contents=[mapping_prompt]
        ).text
        print(f"Mapping response: {response}")
        raw = extract_json(response)
        return json.loads(raw)
    except Exception as e:
        print(f"Error in mapping: {e}")
        return {}

def fill_pdf_form(pdf_bytes, field_mappings):
    """Fill the PDF form with mapped values"""
    doc = fitz.open(stream=pdf_bytes, filetype='pdf')

    filled_count = 0
    total_fields = 0

    for page_num, page in enumerate(doc):
        for widget in page.widgets() or []:
            total_fields += 1
            field_name = widget.field_name

            if field_name in field_mappings and field_mappings[field_name]:
                try:
                    value = field_mappings[field_name]

                    if widget.field_type == fitz.PDF_WIDGET_TYPE_CHECKBOX:
                        if str(value).lower() in ['yes', 'true', '1', 'checked']:
                            widget.field_value = True
                            filled_count += 1
                        elif str(value).lower() in ['no', 'false', '0', 'unchecked']:
                            widget.field_value = False
                            filled_count += 1
                    else:
                        widget.field_value = str(value)
                        filled_count += 1

                    widget.update()

                except Exception as e:
                    print(f"Error filling field {field_name}: {e}")

    output_buffer = BytesIO()
    doc.save(output_buffer)
    doc.close()

    print(f"Filled {filled_count} out of {total_fields} fields")
    return output_buffer.getvalue(), filled_count, total_fields

def validate_and_review_mappings(field_mappings, field_contexts, patient_info):
    """Validate mappings and provide review summary"""
    validation_results = {
        'high_confidence': [],
        'medium_confidence': [],
        'low_confidence': [],
        'unmapped_fields': []
    }

    for field_context in field_contexts:
        field_name = field_context.get('name', '')
        field_question = field_context.get('question', '')
        field_context_text = field_context.get('context', '')

        if field_name in field_mappings and field_mappings[field_name]:
            confidence = 'medium'
            value = field_mappings[field_name]

            if any(keyword in field_question.lower() for keyword in ['name', 'date', 'id', 'phone']):
                confidence = 'high'
            elif any(keyword in field_question.lower() for keyword in ['diagnosis', 'medication', 'dosage', 'clinical']):
                if len(str(value)) < 3:
                    confidence = 'low'

            validation_results[f'{confidence}_confidence'].append({
                'field': field_name,
                'question': field_question,
                'value': value,
                'context': field_context_text
            })
        else:
            validation_results['unmapped_fields'].append({
                'field': field_name,
                'question': field_question,
                'context': field_context_text
            })

    return validation_results

# Streamlit configuration for Colab
st.set_page_config(
    page_title="PAFill - Medical Form Automation",
    page_icon="🏥",
    layout="wide",
    initial_sidebar_state="expanded"
)

def main():
    st.title("🏥 PAFill - Automate Insurance Forms")
    st.write("Upload medical referral packages and Prior Authorization forms to automatically extract and fill patient information.")

    # API Key configuration in sidebar
    st.sidebar.header("⚙️ Configuration")

    # Allow API key input in Colab
    api_key_input = st.sidebar.text_input(
        "Gemini API Key",
        value=API_KEY if API_KEY != "your_api_key_here" else "",
        type="password",
        help="Enter your Google Gemini API key"
    )

    if api_key_input and api_key_input != API_KEY:
        global client
        try:
            client = genai.Client(api_key=api_key_input)
            st.sidebar.success("✅ API Key updated successfully")
        except Exception as e:
            st.sidebar.error(f"❌ API Key error: {e}")
            client = None

    # Processing options
    st.sidebar.header("🔧 Processing Options")
    review_mode = st.sidebar.checkbox("Review mappings before filling", value=True)
    confidence_threshold = st.sidebar.slider("Confidence threshold", 0.0, 1.0, 0.7)

    # File uploaders
    col1, col2 = st.columns(2)

    with col1:
        st.subheader("📄 PA Form")
        pa_file = st.file_uploader("Upload PA form PDF", type=["pdf"], key="pa_form")

    with col2:
        st.subheader("📋 Referral Package")
        ref_file = st.file_uploader("Upload Referral Package PDF", type=["pdf"], key="ref_package")

    if st.button("🚀 Process and Fill Forms", type="primary"):
        if not client:
            st.error("❌ Please configure your Gemini API key first")
            st.stop()

        if not pa_file or not ref_file:
            st.error("❌ Please upload both PA form and referral package PDFs")
            st.stop()

        # Progress tracking
        progress_bar = st.progress(0)
        status_text = st.empty()

        try:
            # Read file bytes
            status_text.text("📖 Reading uploaded files...")
            pa_bytes = pa_file.read()
            ref_bytes = ref_file.read()
            progress_bar.progress(10)

            # 1. Extract patient info
            status_text.text("🔍 Extracting patient information from referral package...")
            patient_info = extract_patient_info(ref_bytes, pa_bytes)

            with st.expander("📊 Extracted Patient Information", expanded=True):
                st.json(patient_info)
            progress_bar.progress(30)

            # 2. Extract PA fields
            status_text.text("📝 Analyzing PA form fields...")
            fields = extract_fields_with_positions(pa_bytes)
            st.info(f"🎯 Found {len(fields)} form fields")

            # Group fields by page
            fields_by_page = {}
            for f in fields:
                fields_by_page.setdefault(f["page"], []).append({
                    "name": f["name"],
                    "type": f["type"],
                    "rect": f["rect"],
                })
            progress_bar.progress(50)

            # 3. Get field contexts page by page
            status_text.text("🧠 Extracting field contexts...")
            all_field_contexts = []

            for page_no, page_fields in sorted(fields_by_page.items()):
                page_contexts = get_field_context_for_page(pa_bytes, page_no, page_fields)
                all_field_contexts.extend(page_contexts)

            st.info(f"🎯 Extracted contexts for {len(all_field_contexts)} fields")
            progress_bar.progress(70)

            # 4. Map patient info to fields
            status_text.text("🔗 Mapping patient information to form fields...")
            field_mappings = map_patient_info_to_fields(patient_info, all_field_contexts)
            progress_bar.progress(85)

            # 5. Validate mappings
            validation_results = validate_and_review_mappings(field_mappings, all_field_contexts, patient_info)

            if review_mode:
                st.subheader("🔍 Field Mapping Review")

                col1, col2, col3 = st.columns(3)

                with col1:
                    if validation_results['high_confidence']:
                        st.success(f"**✅ High Confidence ({len(validation_results['high_confidence'])})**")
                        for item in validation_results['high_confidence']:
                            st.write(f"• {item['field']}: {item['question']} → **{item['value']}**")

                with col2:
                    if validation_results['medium_confidence']:
                        st.warning(f"**⚠️ Medium Confidence ({len(validation_results['medium_confidence'])})**")
                        for item in validation_results['medium_confidence']:
                            st.write(f"• {item['field']}: {item['question']} → *{item['value']}*")

                with col3:
                    if validation_results['low_confidence']:
                        st.error(f"**❌ Low Confidence ({len(validation_results['low_confidence'])})**")
                        for item in validation_results['low_confidence']:
                            st.write(f"• {item['field']}: {item['question']} → ⚠️ *{item['value']}*")

                if validation_results['unmapped_fields']:
                    with st.expander(f"❓ Unmapped Fields ({len(validation_results['unmapped_fields'])})"):
                        for item in validation_results['unmapped_fields']:
                            st.write(f"• {item['field']}: {item['question']}")

                if not st.button("✅ Proceed with Form Filling", type="primary"):
                    st.stop()

            # 6. Fill the PDF form
            status_text.text("📝 Filling PDF form...")
            filled_pdf_bytes, filled_count, total_fields = fill_pdf_form(pa_bytes, field_mappings)
            progress_bar.progress(100)

            # 7. Results and download
            status_text.text("✅ Processing complete!")

            st.success(f"🎉 Successfully filled {filled_count} out of {total_fields} form fields!")

            # Summary statistics
            col1, col2, col3 = st.columns(3)
            with col1:
                st.metric("📝 Fields Filled", filled_count)
            with col2:
                st.metric("📊 Total Fields", total_fields)
            with col3:
                fill_rate = (filled_count / total_fields * 100) if total_fields > 0 else 0
                st.metric("📈 Fill Rate", f"{fill_rate:.1f}%")

            # Download button for filled form
            st.download_button(
                label="📥 Download Filled PA Form",
                data=filled_pdf_bytes,
                file_name=f"filled_pa_form_{datetime.now().strftime('%Y%m%d_%H%M%S')}.pdf",
                mime="application/pdf",
                type="primary"
            )

        except Exception as e:
            st.error(f"❌ An error occurred during processing: {str(e)}")
            with st.expander("🔍 Full error details"):
                st.code(traceback.format_exc())

if __name__ == "__main__":
    main()

Writing app.py


In [6]:
!npm install localtunnel

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋
added 22 packages in 2s
⠋
⠋3 packages are looking for funding
⠋  run `npm fund` for details
⠋

In [ ]:
!streamlit run app.py &>/content/logs.txt & npx localtunnel --port 8501 & curl ipv4.icanhazip.com

34.105.123.227
⠙⠹your url is: https://clean-wombats-strive.loca.lt
